IMPORT LIBRARY

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

from pyspark.ml.feature import (
    RegexTokenizer,
    CountVectorizer,
    IDF
)

from pyspark.sql.types import ArrayType, StringType
from pyspark.ml.stat import Summarizer

import pandas as pd
import numpy as np
from pathlib import Path
from deep_translator import GoogleTranslator
import os 

In [2]:
BASE_DIR = Path.cwd()

while BASE_DIR.name != "Final_Project":

    BASE_DIR = BASE_DIR.parent


os.chdir(BASE_DIR)

print(os.getcwd())

FINAL_DIR = BASE_DIR / "data" / "final"
FINAL_DIR.mkdir(parents=True,exist_ok=True)

c:\Users\sever\Documents\Project_Coolyeah\Lab_MCI\Final_Project


In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("TFIDFAnalysis") \
    .getOrCreate()

DATA LOADING

In [4]:
cleaned_master_df = spark.read.parquet(
    "data/final/master_cleaned"
)

cleaned_master_df.limit(20).toPandas()

,order_id,review_score,review_comment_message,sentiment,product_category_name_english,delivery_delay_days,delivery_status,customer_state,seller_id,seller_state,same_state_shipping
0,10c3c7fea20a1ed15c6e55e90b3fb084,1,Não recebi o produto e nem meu dinheiro de volta,negative,perfumery,17,late,SP,3c7c4a49ec3c6550809089c6a2ca9370,SP,1
1,a3e4fb1d4c4c04deb731bae74f190339,5,Produto de ótima qualidade e chegou antes do p...,positive,sports_leisure,-21,early,AP,376a891762bbdecbc02b4b6adec3fdda,GO,0
2,d7e87a1388d89fa6d7841f32c6954c2e,1,Foram pedidos 2 unidades e recebemos só 1,negative,health_beauty,-25,early,SP,8444e55c1f13cd5c179851e5ca5ebd00,MG,0
3,741bffa3ccd9486d8050c172ea085d43,5,no_review,positive,watches_gifts,-34,early,PR,d921b68bf747894be13a97ae52b0f386,MG,0
4,c6eb7d8978a0d7a7e592ef611e4daf04,5,no_review,positive,fashion_underwear_beach,-19,early,MG,a6fe7de3d16f6149ffe280349a8535a0,SP,0
5,9e51155c4d15c5f62e0b309826c25a02,5,no_review,positive,garden_tools,-2,early,RJ,1f50f920176fa81dab994f9023523100,SP,0
6,20d07181f8cec8c3fa51212c7dd926ea,5,no_review,positive,cool_stuff,-38,early,ES,8444e55c1f13cd5c179851e5ca5ebd00,MG,0
7,dae4fcb62b78ccf2ff3ac4d49113a8f5,5,super recomendo produto profissional de qualidade,positive,housewares,-17,early,CE,aced59e9b31ef866a94f9e7f29d8d418,SP,0
8,4e839b1ce670c701bd83a9d9aeb58969,5,no_review,positive,sports_leisure,-14,early,SP,41da412d33e8da4f22baf55cb1bde82c,ES,0
9,620b3433404a08b299e50c716866a4d1,5,no_review,positive,cool_stuff,-13,early,PE,7a67c85e85bb2ce8582c35f2203ad736,SP,0


DATA SELECTION

In [5]:
negative_reviews_df = cleaned_master_df.filter(
    col("review_score") <= 2
)

DATA CLEANING

In [6]:
print(f"Old data total: {negative_reviews_df.count()}")
negative_reviews_df = negative_reviews_df.filter(
    col("review_comment_message") != "no_review"
)

negative_reviews_df = negative_reviews_df.filter(
    trim(col("review_comment_message")) != ""
)

print(f"Cleaned data total: {negative_reviews_df.count()}")
negative_reviews_df.limit(10).toPandas()

Old data total: 12637
Cleaned data total: 9570


,order_id,review_score,review_comment_message,sentiment,product_category_name_english,delivery_delay_days,delivery_status,customer_state,seller_id,seller_state,same_state_shipping
0,10c3c7fea20a1ed15c6e55e90b3fb084,1,Não recebi o produto e nem meu dinheiro de volta,negative,perfumery,17,late,SP,3c7c4a49ec3c6550809089c6a2ca9370,SP,1
1,d7e87a1388d89fa6d7841f32c6954c2e,1,Foram pedidos 2 unidades e recebemos só 1,negative,health_beauty,-25,early,SP,8444e55c1f13cd5c179851e5ca5ebd00,MG,0
2,df2d910b8b5e5f461f67043489f9569d,1,o carrinho veio com defeito.,negative,cool_stuff,-10,early,PE,48436dade18ac8b2bce089ec2a041202,SP,0
3,1a292836ced1aa222199c34c883f404e,1,o produto não chegou,negative,stationery,7,late,SP,3d871de0142ce09b7081e2b9d1733cb1,SP,1
4,9c37d94303e95d233a4608dabc34f802,1,Produto diferente da foto.,negative,construction_tools_construction,-13,early,SP,8648b1e89e9b349e32d3741b30ec737e,SP,1
5,33dea0203a24cd38fb9a7f8ee4bbe0ee,2,"Comprei por um capacete no tamanho G, claramen...",negative,auto,-9,early,SP,33ac3e28642ab8bda860a2f693000e78,SP,1
6,3f6da1442aba80bcf61179602dfab9ca,1,Produto com atraso. Não por causa de greve de ...,negative,auto,6,late,BA,3db66a856d18a9cba7c9241fc5221c50,MG,0
7,ad263bc9111c1e0cbb780c4b7c1009e6,2,Correio não entregou na minha residência tive ...,negative,electronics,-5,early,PA,128639473a139ac0f3e5f5ade55873a5,PR,0
8,312bfb221da51864b9fc4b361542ee5b,1,Produto chegou com risco e manchado.,negative,watches_gifts,-20,early,SP,966cb4760537b1404caedd472cc610a5,SP,1
9,b73bbac6285251dbf2dd01842e0ce0de,1,Ainda não recebi o produto nem qualquer satisf...,negative,air_conditioning,33,late,RJ,15aac934c58d886785ac1b17953ea898,ES,0


DATA NORMALIZATION

In [7]:
negative_reviews_df = negative_reviews_df.withColumn(
    "review_comment_message",
    trim(
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    lower(
                        col("review_comment_message")
                    ),
                    r"\n|\t", " "
                ),
                r"[^a-zA-ZÀ-ÿ\s]",""
            ),
            r"\s+", " "
        )
    )
)

In [8]:
negative_reviews_df.limit(20).toPandas()

,order_id,review_score,review_comment_message,sentiment,product_category_name_english,delivery_delay_days,delivery_status,customer_state,seller_id,seller_state,same_state_shipping
0,10c3c7fea20a1ed15c6e55e90b3fb084,1,não recebi o produto e nem meu dinheiro de volta,negative,perfumery,17,late,SP,3c7c4a49ec3c6550809089c6a2ca9370,SP,1
1,d7e87a1388d89fa6d7841f32c6954c2e,1,foram pedidos unidades e recebemos só,negative,health_beauty,-25,early,SP,8444e55c1f13cd5c179851e5ca5ebd00,MG,0
2,df2d910b8b5e5f461f67043489f9569d,1,o carrinho veio com defeito,negative,cool_stuff,-10,early,PE,48436dade18ac8b2bce089ec2a041202,SP,0
3,1a292836ced1aa222199c34c883f404e,1,o produto não chegou,negative,stationery,7,late,SP,3d871de0142ce09b7081e2b9d1733cb1,SP,1
4,9c37d94303e95d233a4608dabc34f802,1,produto diferente da foto,negative,construction_tools_construction,-13,early,SP,8648b1e89e9b349e32d3741b30ec737e,SP,1
5,33dea0203a24cd38fb9a7f8ee4bbe0ee,2,comprei por um capacete no tamanho g clarament...,negative,auto,-9,early,SP,33ac3e28642ab8bda860a2f693000e78,SP,1
6,3f6da1442aba80bcf61179602dfab9ca,1,produto com atraso não por causa de greve de c...,negative,auto,6,late,BA,3db66a856d18a9cba7c9241fc5221c50,MG,0
7,ad263bc9111c1e0cbb780c4b7c1009e6,2,correio não entregou na minha residência tive ...,negative,electronics,-5,early,PA,128639473a139ac0f3e5f5ade55873a5,PR,0
8,312bfb221da51864b9fc4b361542ee5b,1,produto chegou com risco e manchado,negative,watches_gifts,-20,early,SP,966cb4760537b1404caedd472cc610a5,SP,1
9,b73bbac6285251dbf2dd01842e0ce0de,1,ainda não recebi o produto nem qualquer satisf...,negative,air_conditioning,33,late,RJ,15aac934c58d886785ac1b17953ea898,ES,0


TEXT PREPROCESSING

In [9]:
tokenizer = RegexTokenizer(
    inputCol="review_comment_message",
    outputCol="words",
    pattern="\\s+"
)

df_token = tokenizer.transform(negative_reviews_df)

df_token.select("review_comment_message", "words").limit(10).toPandas()

,review_comment_message,words
0,não recebi o produto e nem meu dinheiro de volta,"[não, recebi, o, produto, e, nem, meu, dinheir..."
1,foram pedidos unidades e recebemos só,"[foram, pedidos, unidades, e, recebemos, só]"
2,o carrinho veio com defeito,"[o, carrinho, veio, com, defeito]"
3,o produto não chegou,"[o, produto, não, chegou]"
4,produto diferente da foto,"[produto, diferente, da, foto]"
5,comprei por um capacete no tamanho g clarament...,"[comprei, por, um, capacete, no, tamanho, g, c..."
6,produto com atraso não por causa de greve de c...,"[produto, com, atraso, não, por, causa, de, gr..."
7,correio não entregou na minha residência tive ...,"[correio, não, entregou, na, minha, residência..."
8,produto chegou com risco e manchado,"[produto, chegou, com, risco, e, manchado]"
9,ainda não recebi o produto nem qualquer satisf...,"[ainda, não, recebi, o, produto, nem, qualquer..."


TD-IDF IMPLEMENTATION  

In [10]:
cv = CountVectorizer(
    inputCol="words",
    outputCol="tf",
    vocabSize=20000
)
cv_model = cv.fit(df_token)
df_tf = cv_model.transform(df_token)


idf = IDF(inputCol="tf", outputCol="tfidf")
idf_model = idf.fit(df_tf)
df_tfidf = idf_model.transform(df_tf)


idf_values = idf_model.idf.toArray()
vocab = cv_model.vocabulary
threshold = 4.0


important_words = [
    vocab[i] for i, val in enumerate(idf_values)
    if val > threshold
]

print("Important Words Total:", len(important_words))
print("Example:", important_words[:20])

Important Words Total: 9589
Example: ['caixa', 'kit', 'unidades', 'frete', 'relógio', 'valor', 'so', 'data', 'devolver', 'fazer', 'preciso', 'consigo', 'ninguém', 'embalagem', 'correio', 'antes', 'péssima', 'consta', 'demora', 'chegar']


SIGNATURE KEYWORD EXTRACTION

In [11]:
def extract_keywords(tfidf_vector):
    if tfidf_vector is None: return []

    arr = tfidf_vector.toArray()

    nonzero_indices = np.where(arr > 0)[0]

    valid_candidates = [
        i for i in nonzero_indices
        if idf_values[i] > threshold and len(vocab[i]) > 2
    ]
    sorted_keywords = sorted(valid_candidates, key=lambda i: arr[i], reverse=True)

    return [vocab[i] for i in sorted_keywords[:5]]

In [12]:
negative_words = [

    "defeito",
    "quebrado",
    "quebrada",
    "atraso",
    "atrasada",
    "faltando",
    "erro",
    "ruim",
    "péssimo",
    "demora",
    "danificado",
    "problema",
    "cancelado",
    "fraco",
    "sujo",
    "enganado",
    "defeituoso",
    "incompleto",
    "rasgado",
    "amassado",
    "desligado",
    "vazando",
    "errado",
    "trincado",
    "reclamação"

]

In [13]:
df_category_grouped = (

    df_tfidf.groupBy(
        "product_category_name_english"
    )
    .agg(
        Summarizer.mean(
            col("tfidf")
        ).alias("avg_tfidf")
    )
)

category_rows = (
    df_category_grouped.collect()
)

In [14]:
results = []

for row in category_rows:

    category = row[
        "product_category_name_english"
    ]

    tfidf_vector = row[
        "avg_tfidf"
    ]

    arr = tfidf_vector.toArray()

    nonzero_indices = np.where(
        arr > 0
    )[0]

    valid_candidates = [

        i for i in nonzero_indices

        if (

            idf_values[i] > threshold

            and vocab[i] in negative_words
        )
    ]


    sorted_keywords = sorted(

        valid_candidates,

        key=lambda i: arr[i],

        reverse=True
    )


    top_keywords = [

        vocab[i]

        for i in sorted_keywords[:5]
    ]


    results.append({

        "category": category,

        "negative_keywords": top_keywords
    })

result_pd = pd.DataFrame(
    results
)

result_pd.head(20)

,category,negative_keywords
0,art,[fraco]
1,flowers,[]
2,home_construction,"[danificado, demora, amassado, quebrado, ruim]"
3,fashion_male_clothing,[]
4,kitchen_dining_laundry_garden_furniture,"[fraco, erro, atraso]"
5,small_appliances,"[amassado, reclamação, quebrado, erro, quebrada]"
6,la_cuisine,[]
7,bed_bath_table,"[ruim, péssimo, demora, fraco, atraso]"
8,signaling_and_security,"[erro, fraco, quebrado]"
9,office_furniture,"[rasgado, quebrada, reclamação, péssimo, atraso]"


TRANSLATE RESULT

In [15]:
def translate_keywords(keyword_list):

    translated = []

    for word in keyword_list:

        try:

            translated_word = GoogleTranslator(
                source="pt",
                target="en"
            ).translate(word)

            translated.append(translated_word)

        except:

            translated.append(word)

    return translated

In [16]:
translated_result_pd = result_pd.copy()

translated_result_pd["translated_keywords"] = translated_result_pd["negative_keywords"].apply(translate_keywords)

In [17]:
translated_result_pd.head(20)

,category,negative_keywords,translated_keywords
0,art,[fraco],[weak]
1,flowers,[],[]
2,home_construction,"[danificado, demora, amassado, quebrado, ruim]","[damaged, delay, kneaded, broken, bad]"
3,fashion_male_clothing,[],[]
4,kitchen_dining_laundry_garden_furniture,"[fraco, erro, atraso]","[weak, error, delay]"
5,small_appliances,"[amassado, reclamação, quebrado, erro, quebrada]","[kneaded, complaint, broken, error, broken]"
6,la_cuisine,[],[]
7,bed_bath_table,"[ruim, péssimo, demora, fraco, atraso]","[bad, bad, delay, weak, delay]"
8,signaling_and_security,"[erro, fraco, quebrado]","[error, weak, broken]"
9,office_furniture,"[rasgado, quebrada, reclamação, péssimo, atraso]","[scratched, broken, complaint, bad, delay]"


In [18]:
translated_result_pd = translated_result_pd.astype(str)

In [19]:
tfidf_result_df = spark.createDataFrame(
    translated_result_pd[["category", "translated_keywords"]]
)

In [20]:
tfidf_result_df.printSchema()

root
 |-- category: string (nullable = true)
 |-- translated_keywords: string (nullable = true)



In [23]:
os.makedirs("data/final",exist_ok=True)

result_pd.to_csv(
    "data/final/tfidf_keywords.csv",
    index=False
)

print("tfidf_keywords.csv saved")

tfidf_keywords.csv saved
